# Dino Run: Visual Deep Reinforcement Learning

Teaching AI to play the Chrome T-Rex game from raw pixels

Members: DONG WUZHENG  JI DI  XU FANGYUAN  ZHOU ZIQING

**Course**: DSA5102 — Foundations of Machine Learning

**Final Project Group**: Group 31

**Project Option**: Option 2

---

### Group Members

| Member Name |
|-------------|
| *DONG WUZHENG* |
| *JI DI* |
| *XU FANGYUAN* |
| *ZHOU ZIQING* |

---

### Generative AI Usage Declaration

Generative AI tools such as Gemini 4 Argon and ChatGPT 6.1 Sol were used during this project for the following purposes:
- **Code assistance & debugging**: Assisting with code generation and debugging.
- **Writing & organization**: Proofreading, polishing, and structuring Markdown explanations in this notebook and the presentation script.

All experimental design choices, training runs, hyperparameter comparisons, ablation studies, and result interpretations were conducted, executed, and verified by the group members.


## Overview

This project uses deep reinforcement learning to teach an AI agent how to play the Chrome T-Rex Runner game. The agent only sees raw pixel frames from the game — it has no access to speed, distance, or any internal state. From these pixels alone, it needs to figure out where obstacles are, how far away they are, and when to jump.

We set up the game in a headless Firefox browser and made it fully deterministic by freezing the browser clock, fixing game speed, and seeding the random number generator. This way every experiment is reproducible. The agent's input is a stack of 4 grayscale frames resized to 80×80 pixels, and it can choose to either keep running or jump. The reward gives a small bonus for surviving each step, a penalty for crashing, and an extra penalty for dying early.

We trained two main algorithms: DQN, which learns Q-values through experience replay, and PPO, which uses policy gradients with clipped updates. Both use the same CNN architecture and train under the same budget of 2.23 million frames across 4 parallel game instances.

To evaluate our methods, we compared 7 approaches under identical conditions — including random baseline, rule-based heuristic, several value-based methods, and PPO. We also ran an ablation study with 14 variants to understand which components of the system matter most. As an extra exploration, we tried fine-tuning a vision-language model through behavior cloning on expert demonstrations.

## 1. Problem Definition & Motivation


### Game Overview & Experimental Rules

T-Rex Runner is the built-in offline mini-game in Google Chrome: a pixel dinosaur runs automatically from left to right and must jump over incoming obstacles. Hitting any obstacle ends the episode immediately, and the final score depends on survived distance. Although the game looks simple, it forms a complete sequential decision-making problem — observe the world, take an action, and receive a delayed outcome — making it a clean, controllable testbed for vision-based reinforcement learning.

- **Original game vs. experimental setup**: In the original Chrome game, running speed gradually increases and pterodactyls eventually appear at high speed (requiring ducking). To ensure controlled and reproducible comparisons across algorithms, our environment fixes the spead at a constant 6 pixels per frame. At this speed, pterodactyls never spawn and only single or clustered cacti appear.
- **Action & ballistic physics**: The player chooses only whether to jump (`Up`) or keep running. Jumping triggers a fixed parabolic trajectory with no mid-air correction — jumping too early lands on the obstacle, jumping too late collides with it, and landing must finish before the next obstacle arrives. Precise jump timing is therefore essential: while the game engine checks collisions on every frame, the agent must commit to its jumps at discrete decision intervals.

### Input Modalities 

There are two ways to feed game information to an agent: **state-based input** (reading structured numerical variables directly from the game engine) and **visual input** (capturing raw pixel frames from the game canvas). This project mainly focuses on the **visual** setting.

### Challenges of Visual Control

- Only raw pixels are available — the agent must infer obstacle position, width, height, and time-to-contact directly from small grayscale images.
- During core RL training (DQN and PPO), no human teacher or labeled action dataset is provided — the agent must learn binary run/jump decisions purely from survival and collision rewards.
- Rewards are spread across thousands of frames, so the agent must determine which specific jump decisions led to long-term survival or a collision.
- A single static frame shows where the dinosaur and obstacles are, but cannot capture temporal motion dynamics — such as whether the dinosaur is rising or falling mid-jump and how obstacles close in across consecutive frames. Integrating multiple frames is therefore necessary to perceive motion.

### Goal & Random Baseline Preview

Our goal is to train an agent to play T-Rex Runner purely from raw pixel observations and reach a high score. Alongside specialized visual deep RL (DQN variants and PPO), we also explore a pretrained game vision-language model (**GameScaling VLM**) as an alternative visual policy.

**Random policy demo (crashes within seconds):**

**Game screenshot:**


In [ ]:
from IPython.display import Image, display
display(Image('results/pptx_images/slide2_3.jpg'))


**Input types: state-based vs visual (the core challenge):**


In [ ]:
from IPython.display import Image, display
display(Image('results/pptx_images/slide3_5.png'))


**Random policy below:**

In [ ]:
from IPython.display import Image, display

In [ ]:
from IPython.display import Image, display
display(Image('results/123.gif'))

In this demo, the random policy crashes within seconds into the first cactus. Solving this reliably from pixels requires learning precise jump timing.


### Alternative Approach: GameScaling VLM

Beyond task-specific RL, we also explore **GameScaling-0.8B-S3**, a pretrained multimodal language model for games that takes game screenshots as input and outputs key-press actions in a structured action format. Each decision covers a 200-millisecond chunk — twelve game frames — and the action is parsed into per-chunk key groups. The model is queried with a system prompt describing the game rules, plus a multi-turn history of up to five recent frames and its own previous actions, so it can see motion and its own decisions. Everything else in the pipeline — capture, chunking, key execution — is identical to the RL environment.


## 2. Environment, MDP Formulation & Visual Models


### Deterministic Browser Environment

To train efficiently and compare algorithms fairly, we avoid real-time screen capture — which is slow, subject to OS/browser frame jitter, and irreproducible. Instead, we use Selenium to drive a headless Firefox browser (eliminating window-rendering overhead and enabling parallel instances) serving the game page over a local HTTP server. JavaScript injection gives deterministic, faster-than-real-time control:

- Freeze `performance.now` and `requestAnimationFrame` so browser frame timing never drifts
- `window.__dinoAdvance(n)` advances the game frame-by-frame (checking collisions every frame) for `n` frames per action call (`ACTION_REPEAT = 4`, giving the agent one decision every ~67 ms of game time at 60 FPS)
- `ACCELERATION = 0` keeps speed constant at 6 pixels/frame
- `Math.random` is replaced with a seeded linear congruential generator (LCG) so every seed reproduces the exact same obstacle sequence

Import dependencies and configure global constants:

Import dependencies and configure the environment:

In [ ]:
import base64, copy, io, json, math, random, shutil, threading, time
from collections import deque
from http.server import SimpleHTTPRequestHandler, ThreadingHTTPServer
from pathlib import Path
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from PIL import Image as PILImage  # renamed to avoid shadowing IPython.display.Image
from IPython.display import display, HTML
from selenium import webdriver
from selenium.webdriver.firefox.options import Options
from selenium.webdriver.firefox.service import Service

DEVICE = torch.device('cpu')
torch.set_num_threads(1)
ACTION_REPEAT = 4
GAMMA = 0.99
SEED = 20261015
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
print('Imports done')

Define a local HTTP server to serve the game files to Firefox:

In [ ]:
GAME_DIR = Path('t-rex-runner-gh-pages').resolve()
assert (GAME_DIR / 'index.html').is_file()
print('Game folder:', GAME_DIR)

class LocalGameHandler(SimpleHTTPRequestHandler):
    def __init__(self, *a, **k):
        super().__init__(*a, directory=str(GAME_DIR), **k)
    def end_headers(self):
        self.send_header('Cache-Control', 'no-store')
        super().end_headers()
    def log_message(self, *a):
        pass

def start_asset_server():
    global asset_server, asset_thread
    if globals().get('asset_server') is None:
        asset_server = ThreadingHTTPServer(('127.0.0.1', 0), LocalGameHandler)
        asset_thread = threading.Thread(target=asset_server.serve_forever, daemon=True)
        asset_thread.start()
    return 'http://127.0.0.1:' + str(asset_server.server_port) + '/'

print('HTTP server ready')

### Visual MDP Formulation

**Observation Pipeline**
1. Capture the game canvas as PNG base64
2. Convert to grayscale 
3. Crop to the active gameplay area (`500×300`) to remove irrelevant right-side margins
4. Resize to `80×80` pixels — balancing obstacle legibility against CNN compute cost 
5. Stack the most recent **4 frames** into a `4×80×80` tensor. Four frames give the network motion cues: from a single still image the network cannot tell whether an obstacle is approaching or receding, but across four frames the displacement is directly visible.

**Action Space**
- `0` = keep running (noop)
- `1` = jump
- Ducking is not modeled because in our constant-speed configuration pterodactyls only appear at jumpable heights or high above the head. Two actions also make the Q-function easy to estimate and the policy easy to visualize

**Reward Design**
- Each surviving step earns a small positive reward of plus `0.1`. 
- A collision gives `-1`. 
- If the episode ends with a score below `1000`, an extra `-5` penalty is applied. The shaped step reward densifies the signal so the agent learns to survive; the crash penalty makes collisions costly; and the early-crash penalty specifically discourages suicidal jumping — without it the agent can farm step reward by jumping randomly into obstacles early, which looks safe locally but never learns to play.
- Discount factor: `gamma = 0.99`

**Action space and reward diagrams:**

In [ ]:
from IPython.display import Image, display
display(Image('results/pptx_images/slide9_8.png'))

In [ ]:
from IPython.display import Image, display
display(Image('results/pptx_images/slide9_9.png'))

In [ ]:
from IPython.display import Image, display
display(Image('results/pptx_images/slide10_3.png'))

### Visual RL Network Architecture

The vision network is a convolutional neural net we call GitHubDinoNet. Three convolutional layers with max pooling extract spatial features from the stacked frames. The feature map is flattened, passed through a 512-unit fully connected layer with ReLU, and projected to 2 Q-values — one per action. CNN is the right choice here because obstacle position, shape, and height are spatial features, and convolution is translation-aware by construction. The model is small — on the order of a hundred thousand parameters — so it trains fast on a single machine.

On top of the architecture, two mechanisms matter for training stability. Frame stacking groups the latest 4 frames together as described above. Prioritized experience replay samples transitions with probability proportional to their temporal-difference error, so surprising transitions — the ones the network has not yet mastered — are seen more often. The replay buffer is capped and uses importance weights to correct the sampling bias.

```
Input: 4 x 80 x 80 grayscale frame stack
  -> Conv(4, 32, 8x8, s4) + ReLU + MaxPool(2,2)
  -> Conv(32, 64, 4x4, s2) + ReLU + MaxPool(2,2)
  -> Conv(64, 64, 3x3, s1) + ReLU + MaxPool(2,2)
  -> Flatten(64)
  -> FC(64, 512) + ReLU
  -> FC(512, 2)
Output: 2 Q-values (run, jump)
```


**Overall RL workflow:**

In [ ]:
from IPython.display import Image, display
display(Image('results/pptx_images/slide5_5.png'))

**State-based agent architecture:**

In [ ]:
from IPython.display import Image, display
display(Image('results/pptx_images/slide6_3.png'))

**Visual agent architecture:**

In [ ]:
from IPython.display import Image, display
display(Image('results/pptx_images/slide7_3.png'))

**GameScaling VLM policy:**

GameScaling-0.8B-S3 is a multimodal language model that takes game screenshots as input and outputs key-press actions in a structured action format. Each decision covers a 200ms chunk (12 game frames). The model uses a system prompt with game rules plus multi-turn history (up to 5 recent frames and previous actions), so it can observe motion and its own past decisions. Capture, chunking, and key execution match the RL environment.


In [ ]:
from IPython.display import Image, display
display(Image('results/pptx_images/slide4_5.png'))

Define the network and replay buffer:

In [ ]:
class GitHubDinoNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(4, 32, 8, stride=4, padding=0)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 4, stride=2, padding=0)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 64, 3, stride=1, padding=0)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64, 512)
        self.fc2 = nn.Linear(512, 2)

    @staticmethod
    def _keras_same(conv, x):
        h, w = x.shape[-2:]
        kh, kw = conv.kernel_size
        sh, sw = conv.stride
        oh, ow = (h + sh - 1) // sh, (w + sw - 1) // sw
        ph, pw = max((oh - 1) * sh + kh - h, 0), max((ow - 1) * sw + kw - w, 0)
        x = F.pad(x, (pw // 2, pw - pw // 2, ph // 2, ph - ph // 2))
        return conv(x)

    def forward(self, x):
        x = F.relu(self.pool1(self._keras_same(self.conv1, x)))
        x = F.relu(self.pool2(self._keras_same(self.conv2, x)))
        x = F.relu(self.pool3(self._keras_same(self.conv3, x)))
        x = x.permute(0, 2, 3, 1).contiguous().flatten(1)
        return self.fc2(F.relu(self.fc1(x)))

net = GitHubDinoNet()
print('Parameters:', sum(p.numel() for p in net.parameters()))
print(net)

Frame stacking groups the latest 4 frames together. Prioritized replay samples transitions by TD error, focusing training on harder cases.

In [ ]:
def stack_frames(frames):
    return np.stack(list(frames)[::-1], axis=0)

class PrioritizedReplay:
    def __init__(self, cap=50000, alpha=0.6):
        self.items = []
        self.priorities = []
        self.cap = cap
        self.pos = 0
        self.alpha = alpha

    def add(self, item):
        p = max(self.priorities, default=1.0)
        if len(self.items) < self.cap:
            self.items.append(item)
            self.priorities.append(p)
        else:
            self.items[self.pos] = item
            self.priorities[self.pos] = p
            self.pos = (self.pos + 1) % self.cap

    def sample(self, n, beta):
        p = np.asarray(self.priorities, dtype=np.float64) ** self.alpha
        p /= p.sum()
        ids = np.random.choice(len(self.items), n, p=p)
        w = (len(self.items) * p[ids]) ** (-beta)
        w /= w.max()
        return [self.items[i] for i in ids], ids, w.astype(np.float32)

    def update(self, ids, errors):
        for i, e in zip(ids, errors):
            self.priorities[int(i)] = float(abs(e) + 1e-5)

replay = PrioritizedReplay(cap=2000)
print('Replay buffer ready')

### Game Environment (LocalGitHubGame)

This class wraps the browser game into a standard RL interface:

- `reset(seed)`: loads page, injects JS, seeds RNG, starts a new game
- `step(action)`: applies action (run/jump), advances 4 frames, returns (observation, reward, done, score)

In [ ]:
class LocalGitHubGame:
    def __init__(self, seed, action_repeat=ACTION_REPEAT, gamma=GAMMA):
        self.action_repeat = max(1, int(action_repeat))
        self.gamma = float(gamma)
        self.url = start_asset_server()
        options = Options()
        options.add_argument('-headless')
        options.set_preference('media.volume_scale', '0.0')
        self.driver = webdriver.Firefox(options=options)
        self.driver.set_window_size(1000, 700)

    def reset(self, seed):
        self.driver.get(self.url)
        deadline = time.monotonic() + 15
        while not self.driver.execute_script('return !!(window.dinoControl && window.dinoControl.getState().ready)'):
            if time.monotonic() > deadline:
                raise RuntimeError('Game init timeout')
            time.sleep(.05)
        self.driver.execute_script("const el=document.createElement('script');el.textContent='(()=>{let s='+String(arguments[0]>>>0)+'; Math.random=()=>{s=(1664525*s+1013904223)>>>0;return s/4294967296;};})();';document.head.appendChild(el);el.remove();", int(seed))
        self.driver.execute_script("const r=Runner.instance_;r.stop();Runner.config.ACCELERATION=0;r.config.ACCELERATION=0;let t=performance.now();Object.defineProperty(performance,'now',{configurable:true,value:()=>t});window.requestAnimationFrame=()=>1;window.cancelAnimationFrame=()=>{};let introStart=null;window.__dinoAdvance=(n)=>{let advanced=0;for(let i=0;i<n && r.playing && !r.crashed;i++){if(r.playingIntro && introStart===null)introStart=t;if(r.playingIntro && t-introStart>=400){r.startGame();introStart=null;}t+=1000/60;r.update();advanced++;if(!r.playingIntro)introStart=null;}return {state:window.dinoControl.getState(),advanced,frame:document.querySelector('canvas.runner-canvas').toDataURL('image/png').split(',')[1]};};r.time=t;const e={keyCode:38,target:document.body,preventDefault(){}};r.onKeyDown(e);r.onKeyUp(e);")
        return self.observe()

    @staticmethod
    def _process_frame(encoded):
        img = PILImage.open(io.BytesIO(base64.b64decode(encoded))).convert('L')
        img = img.crop((0, 0, min(500, img.width), min(300, img.height)))
        return np.asarray(img.resize((80, 80), PILImage.Resampling.BILINEAR), dtype=np.uint8)

    def observe(self):
        encoded = self.driver.execute_script("return document.querySelector('canvas.runner-canvas').toDataURL('image/png').split(',')[1]")
        return self._process_frame(encoded)

    def step(self, action):
        if action not in (0, 1):
            raise ValueError('Actions: 0=run, 1=jump')
        if action == 1:
            self.driver.execute_script("const r=Runner.instance_;if(!r.tRex.jumping){const e={keyCode:38,target:document.body,preventDefault(){}};r.onKeyDown(e);r.onKeyUp(e);}")
        result = self.driver.execute_script('return window.__dinoAdvance(arguments[0])', self.action_repeat)
        done = bool(result['state']['crashed'])
        advanced = int(result['advanced'])
        reward = sum(self.gamma**i * (-1.0 if done and i == advanced-1 else .1) for i in range(advanced))
        return (self._process_frame(result['frame']), reward, done, float(result['state']['score']), self.gamma**advanced, advanced)

    def close(self):
        self.driver.quit()

print('LocalGitHubGame ready')

## 3. Training

### Algorithms

**DQN (Deep Q-Network)**
- Neural net approximates Q-values for each action
- Double DQN: online net selects action, target net evaluates it
- Prioritized replay: sample by TD error
- Slow target network update (tau = 0.001)

**PPO (Proximal Policy Optimization)**
- Policy gradient method with clipped updates (clip = 0.2)
- Generalized advantage estimation (lambda = 0.95)
- Entropy bonus prevents premature convergence (coef = 0.01)
- Parallel environments for faster data collection

### Training Parameters

- Budget: 2.23 million frames
- Parallel environments: 4
- DQN: lr=3e-5, batch=64, epsilon 0.10 -> 0.02
- PPO: lr=2.5e-4, clip=0.2, epochs=4

### DQN Training

Set `RUN_DQN_TRAINING = True` to start training. Set to `False` to skip and use the pretrained checkpoint.

Initialize the DQN network and optimizer:

In [ ]:
RUN_DQN_TRAINING = False

TRAIN_SECONDS = 120
TRAIN_EPISODES = 100
MAX_GAME_FRAMES = 20000
BATCH_SIZE = 64
EARLY_CRASH_PENALTY = 5.0
STABILITY_SCORE = 1000

learner = GitHubDinoNet().to(DEVICE)
target = copy.deepcopy(learner).eval().requires_grad_(False)
optimizer = torch.optim.Adam(learner.parameters(), lr=3e-5)
training_history = []
updates = 0
print('DQN setup done')

The DQN training loop: each episode uses epsilon-greedy action selection, stores transitions in replay, and periodically samples batches to update the Q network.

In [ ]:
def train_dqn():
    global training_history, updates
    game = LocalGitHubGame(SEED)
    started = time.monotonic()
    total_frames = 0
    for episode in range(1, TRAIN_EPISODES + 1):
        if time.monotonic() - started >= TRAIN_SECONDS:
            break
        actor = copy.deepcopy(learner).eval().requires_grad_(False)
        epsilon = max(.02, .10 - .08 * total_frames / 1000000)
        obs = game.reset(SEED + episode)
        frames = deque([obs.copy() for _ in range(4)], maxlen=4)
        state = stack_frames(frames)
        done = False
        score = 0.
        game_frames = 0
        while not done and game_frames < MAX_GAME_FRAMES:
            with torch.inference_mode():
                greedy = int(actor(torch.from_numpy(state).float().unsqueeze(0)).argmax(1).item())
            action = random.randrange(2) if random.random() < epsilon else greedy
            image, reward, done, score, discount, advanced = game.step(action)
            frames.append(image)
            nxt = stack_frames(frames)
            if done and score < STABILITY_SCORE:
                reward -= EARLY_CRASH_PENALTY * GAMMA ** max(0, advanced - 1)
            replay.add((state.copy(), action, reward, nxt.copy(), done, discount))
            state = nxt
            game_frames += advanced
            total_frames += advanced
            if len(replay.items) >= max(100, BATCH_SIZE):
                batch, ids, weights = replay.sample(BATCH_SIZE, min(1., .4 + .6 * updates / 100000))
                st = torch.from_numpy(np.stack([z[0] for z in batch])).float()
                ns = torch.from_numpy(np.stack([z[3] for z in batch])).float()
                acts = torch.tensor([z[1] for z in batch], dtype=torch.long)
                rews = torch.tensor([z[2] for z in batch], dtype=torch.float32)
                terms = torch.tensor([z[4] for z in batch], dtype=torch.float32)
                disc = torch.tensor([z[5] for z in batch], dtype=torch.float32)
                q = learner(st).gather(1, acts[:, None]).squeeze(1)
                with torch.no_grad():
                    na = learner(ns).argmax(1)
                    expected = rews + disc * (1 - terms) * target(ns).gather(1, na[:, None]).squeeze(1)
                errors = (q - expected).detach().abs().numpy()
                loss = (torch.from_numpy(weights) * F.smooth_l1_loss(q, expected, reduction='none')).mean()
                optimizer.zero_grad()
                loss.backward()
                torch.nn.utils.clip_grad_norm_(learner.parameters(), 10)
                optimizer.step()
                with torch.no_grad():
                    for tp, lp in zip(target.parameters(), learner.parameters()):
                        tp.lerp_(lp, .001)
                replay.update(ids, errors)
                updates += 1
        training_history.append({'episode': episode, 'score': score, 'epsilon': epsilon})
        print('ep=%d score=%.0f eps=%.3f' % (episode, score, epsilon))
    game.close()

print('train_dqn defined')

In [ ]:
# Unified off-policy trainer: DQN / Double-DQN / C51, vision or state observations.
# This is the complete training loop used for all comparison and ablation experiments.
import time as _time

V_MIN, V_MAX, N_ATOMS = -20.0, 20.0, 51
ATOMS = torch.linspace(V_MIN, V_MAX, N_ATOMS)

def c51_project(rewards, discounts, terminals, next_probs):
    """Project Bellman target onto fixed categorical support (C51)."""
    b = rewards[:, None] + (discounts * (1 - terminals))[:, None] * ATOMS[None, :]
    b = b.clamp(V_MIN, V_MAX)
    idx = (b - V_MIN) / ((V_MAX - V_MIN) / (N_ATOMS - 1))
    l = idx.floor().long().clamp(0, N_ATOMS - 1)
    u = idx.ceil().long().clamp(0, N_ATOMS - 1)
    eq = l == u
    l = torch.where(eq & (u > 0), l - 1, l)
    u = torch.where(eq & (u < N_ATOMS - 1), u + 1, u)
    w_l = next_probs * (u.float() - idx)
    w_u = next_probs * (idx - l.float())
    proj = torch.zeros_like(next_probs)
    proj.scatter_add_(1, l, w_l)
    proj.scatter_add_(1, u, w_u)
    return proj

def train_offpolicy(algo='ddqn', obs='vision', name=None, frames_budget=2_230_000,
                    feat_mask=None, frame_stack=4, obs_res=80,
                    step_reward=0.1, crash_penalty=1.0, early_penalty=5.0,
                    seed=20261015, lr=3e-4, batch_size=256):
    """Train DQN/DDQN/C51 on the T-Rex game.

    Args:
        algo: 'dqn' (vanilla), 'ddqn' (Double-DQN), or 'c51' (categorical).
        obs: 'vision' (frame stack) or 'state' (26-dim engine features).
        feat_mask: optional feature mask for state obs ('distonly', 'notrex', etc.).
    Returns: dict with training history and final model.
    """
    name = name or f'{algo}_{obs}'
    out_dir = Path('results/exp') / name
    out_dir.mkdir(parents=True, exist_ok=True)
    log_path = out_dir / 'training_log.csv'

    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)

    is_c51 = algo == 'c51'
    out_dim = 2 * N_ATOMS if is_c51 else 2
    use_state = (obs == 'state')

    # Network: vision uses GitHubDinoNet; state uses a small MLP
    if use_state:
        feat_dim = 26
        if feat_mask == 'distonly': feat_dim = 4
        elif feat_mask == 'notrex': feat_dim = 21
        elif feat_mask == 'notype': feat_dim = 17
        elif feat_mask == 'noheight': feat_dim = 17
        class StateNet(nn.Module):
            def __init__(self):
                super().__init__()
                self.net = nn.Sequential(
                    nn.Linear(feat_dim, 128), nn.ReLU(),
                    nn.Linear(128, 128), nn.ReLU(),
                    nn.Linear(128, out_dim))
            def forward(self, x): return self.net(x)
        learner = StateNet().to(DEVICE)
    else:
        if is_c51:
            class C51Net(nn.Module):
                def __init__(self):
                    super().__init__()
                    self.conv1 = nn.Conv2d(frame_stack, 32, 8, stride=4)
                    self.pool1 = nn.MaxPool2d(2, 2)
                    self.conv2 = nn.Conv2d(32, 64, 4, stride=2)
                    self.pool2 = nn.MaxPool2d(2, 2)
                    self.conv3 = nn.Conv2d(64, 64, 3, stride=1)
                    self.pool3 = nn.MaxPool2d(2, 2)
                    self.fc1 = nn.Linear(64, 512)
                    self.fc2 = nn.Linear(512, out_dim)
                def forward(self, x):
                    x = F.relu(self.pool1(self.conv1(x)))
                    x = F.relu(self.pool2(self.conv2(x)))
                    x = F.relu(self.pool3(self.conv3(x)))
                    x = x.permute(0, 2, 3, 1).contiguous().flatten(1)
                    return self.fc2(F.relu(self.fc1(x)))
            learner = C51Net().to(DEVICE)
        else:
            learner = GitHubDinoNet().to(DEVICE)

    target = copy.deepcopy(learner).eval().requires_grad_(False)
    optimizer = torch.optim.Adam(learner.parameters(), lr=lr)
    replay = PrioritizedReplay(cap=2000)
    history = []
    updates = 0
    total_frames = 0
    best_score = 0
    game = LocalGitHubGame(SEED, action_repeat=4)
    started = _time.monotonic()
    episode = 0

    while total_frames < frames_budget and _time.monotonic() - started < 3600:
        episode += 1
        actor = copy.deepcopy(learner).eval().requires_grad_(False)
        epsilon = max(.02, .10 - .08 * total_frames / 1_000_000)
        seed_i = SEED + episode
        obs_raw = game.reset(seed_i)
        frames = deque([obs_raw.copy() for _ in range(frame_stack)], maxlen=frame_stack)
        if use_state:
            state = game.get_state() if hasattr(game, 'get_state') else np.zeros(feat_dim, dtype=np.float32)
        else:
            state = stack_frames(frames)
        done = False; score = 0.; game_frames = 0; losses = []

        while not done and game_frames < 20000:
            with torch.inference_mode():
                q = actor(torch.from_numpy(state).float().unsqueeze(0) if not use_state
                          else torch.from_numpy(state).float().unsqueeze(0))
                greedy = int(q.argmax(1).item())
            action = random.randrange(2) if random.random() < epsilon else greedy
            image, reward, done, score, discount, advanced = game.step(action)
            frames.append(image)
            if use_state:
                nxt = game.get_state() if hasattr(game, 'get_state') else np.zeros(feat_dim, dtype=np.float32)
            else:
                nxt = stack_frames(frames)
            r = step_reward * advanced
            if done: r = -crash_penalty
            if done and score < 1000: r -= early_penalty
            replay.add((state.copy(), action, r, nxt.copy(), done, discount))
            state = nxt; game_frames += advanced; total_frames += advanced

            if len(replay.items) >= 100:
                batch, ids, weights = replay.sample(batch_size, min(1., .4 + .6 * updates / 100000))
                st = torch.from_numpy(np.stack([z[0] for z in batch])).float()
                ns = torch.from_numpy(np.stack([z[3] for z in batch])).float()
                acts = torch.tensor([z[1] for z in batch], dtype=torch.long)
                rews = torch.tensor([z[2] for z in batch], dtype=torch.float32)
                terms = torch.tensor([z[4] for z in batch], dtype=torch.float32)
                disc = torch.tensor([z[5] for z in batch], dtype=torch.float32)

                if is_c51:
                    logits = learner(st).view(-1, 2, N_ATOMS)
                    log_probs = F.log_softmax(logits, dim=2)
                    q_dist = log_probs[range(len(acts)), acts]
                    with torch.no_grad():
                        next_logits = learner(ns).view(-1, 2, N_ATOMS)
                        next_probs = F.softmax(next_logits, dim=2)
                        if algo == 'ddqn':
                            na = next_logits.sum(2).argmax(1)
                        else:
                            na = next_probs.sum(2).argmax(1)
                        target_probs = next_probs[range(len(acts)), na]
                        proj = c51_project(rews, disc, terms, target_probs)
                    loss = -(proj * q_dist).sum(1).mean()
                else:
                    q = learner(st).gather(1, acts[:, None]).squeeze(1)
                    with torch.no_grad():
                        if algo == 'ddqn':
                            na = learner(ns).argmax(1)
                            expected = rews + disc * (1 - terms) * target(ns).gather(1, na[:, None]).squeeze(1)
                        else:
                            expected = rews + disc * (1 - terms) * target(ns).max(1).values
                    errors = (q - expected).detach().abs().numpy()
                    loss = (torch.from_numpy(weights) * F.smooth_l1_loss(q, expected, reduction='none')).mean()

                optimizer.zero_grad(); loss.backward()
                torch.nn.utils.clip_grad_norm_(learner.parameters(), 10); optimizer.step()
                with torch.no_grad():
                    for tp, lp in zip(target.parameters(), learner.parameters()):
                        tp.lerp_(lp, .001)
                if not is_c51:
                    replay.update(ids, errors)
                updates += 1; losses.append(float(loss))

        history.append(dict(episode=episode, score=score, total_frames=total_frames,
                           epsilon=epsilon, mean_loss=float(np.mean(losses)) if losses else 0))
        if score > best_score:
            best_score = score
            torch.save({'model': actor.state_dict(), 'score': score, 'episode': episode, 'algo': algo},
                       out_dir / 'best.pt')
        if episode % 50 == 0:
            print(f'  {name} ep={episode} score={score:.0f} best={best_score:.0f} frames={total_frames}')

    game.close()
    torch.save({'model': learner.state_dict(), 'score': best_score, 'algo': algo},
               out_dir / 'final.pt')
    # Write training log
    with open(log_path, 'w', newline='') as f:
        w = csv.DictWriter(f, fieldnames=['episode', 'score', 'total_frames', 'epsilon', 'mean_loss'])
        w.writeheader(); w.writerows(history)
    print(f'{name}: {episode} episodes, best={best_score:.0f}, frames={total_frames}, -> {out_dir}')
    return dict(history=history, best=best_score, name=name)

print('train_offpolicy defined — supports dqn/ddqn/c51 x vision/state')

Run training:

In [ ]:
if RUN_DQN_TRAINING:
    train_dqn()
else:
    # Load pretrained DQN checkpoint
    CHECKPOINT = Path('dino_model.pt')
    if CHECKPOINT.is_file():
        checkpoint_bytes = CHECKPOINT.read_bytes()
        saved = torch.load(io.BytesIO(checkpoint_bytes), map_location=DEVICE, weights_only=True)
        learner.load_state_dict(saved['model'])
        target = copy.deepcopy(learner).eval().requires_grad_(False)
        learner.eval().requires_grad_(False)
        print(f'Loaded pretrained DQN checkpoint: {CHECKPOINT.name}, episode {saved.get("episodes", "unknown")}')
    else:
        print(f'DQN checkpoint not found: {CHECKPOINT}')


### PPO Training

Set `RUN_PPO_TRAINING = True` to start training.

Configure PPO hyperparameters:

In [ ]:
RUN_PPO_TRAINING = False

PPO_LR = 2.5e-4
PPO_CLIP = 0.2
PPO_EPOCHS = 4
GAE_LAMBDA = 0.95
ENTROPY_COEF = 0.01
VALUE_COEF = 0.5

print('PPO: lr=%s clip=%s lambda=%s entropy=%s' % (PPO_LR, PPO_CLIP, GAE_LAMBDA, ENTROPY_COEF))

Run PPO training (full logic in train_ppo.py):

In [ ]:
RUN_PPO_TRAINING = False

# ===== Real PPO implementation (copied from train_ppo.py / dino_env.py) =====
class PPOVisionNet(nn.Module):
    """Dino conv trunk + policy/value heads. Matches results/exp/ppo_vision/best.pt."""
    def __init__(self, in_channels=4, n_actions=2):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, 32, 8, stride=4)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, 4, stride=2)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.conv3 = nn.Conv2d(64, 64, 3, stride=1)
        self.pool3 = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64, 512)
        self.policy = nn.Linear(512, n_actions)
        self.value = nn.Linear(512, 1)

    @staticmethod
    def _keras_same(conv, x):
        h, w = x.shape[-2:]
        kh, kw = conv.kernel_size
        sh, sw = conv.stride
        oh, ow = (h + sh - 1) // sh, (w + sw - 1) // sw
        ph, pw = max((oh - 1) * sh + kh - h, 0), max((ow - 1) * sw + kw - w, 0)
        x = F.pad(x, (pw // 2, pw - pw // 2, ph // 2, ph - ph // 2))
        return conv(x)

    def forward(self, x):
        x = F.relu(self.pool1(self._keras_same(self.conv1, x)))
        x = F.relu(self.pool2(self._keras_same(self.conv2, x)))
        x = F.relu(self.pool3(self._keras_same(self.conv3, x)))
        x = x.permute(0, 2, 3, 1).contiguous().flatten(1)
        h = F.relu(self.fc1(x))
        return self.policy(h), self.value(h).squeeze(-1)


def compute_gae(rewards, values, discounts, dones, last_values, lam):
    """Per-env GAE. Arrays shaped (T, N). last_values: (N,). Copied from train_ppo.py."""
    T, N = rewards.shape
    adv = np.zeros((T, N), dtype=np.float32)
    last_gae = np.zeros(N, dtype=np.float32)
    for t in reversed(range(T)):
        next_v = last_values if t == T - 1 else values[t + 1]
        next_nonterminal = 1.0 - dones[t]
        delta = rewards[t] + discounts[t] * next_v * next_nonterminal - values[t]
        last_gae = delta + discounts[t] * lam * next_nonterminal * last_gae
        adv[t] = last_gae
    returns = adv + values
    return adv, returns


def train_ppo(total_updates=200, rollout_rounds=64, epochs=4, minibatch=256,
              lr=2.5e-4, clip=0.2, gae_lambda=0.95, entropy_coef=0.01,
              value_coef=0.5, max_grad_norm=0.5, n_envs=2):
    """Real PPO training: rollout -> GAE -> clipped policy gradient update."""
    from torch.distributions import Categorical
    net = PPOVisionNet(in_channels=4).to(DEVICE)
    optimizer = torch.optim.Adam(net.parameters(), lr=lr, eps=1e-5)
    envs = [LocalGitHubGame(SEED + i) for i in range(n_envs)]
    obs = [e.reset(SEED + 1000 + i) for i, e in enumerate(envs)]
    stacks = [deque([o.copy() for _ in range(4)], maxlen=4) for o in obs]
    states = [stack_frames(s) for s in stacks]
    best_score, episode_counter, total_frames, updates = -1.0, 0, 0, 0
    out = Path('results/exp/ppo_vision'); out.mkdir(parents=True, exist_ok=True)
    started = time.monotonic()
    try:
        while updates < total_updates:
            R = rollout_rounds
            ob_s = np.zeros((R, n_envs, 4, 80, 80), dtype=np.uint8)
            acts = np.zeros((R, n_envs), dtype=np.int64)
            rews = np.zeros((R, n_envs), dtype=np.float32)
            discs = np.zeros((R, n_envs), dtype=np.float32)
            dones = np.zeros((R, n_envs), dtype=np.float32)
            logps = np.zeros((R, n_envs), dtype=np.float32)
            vals = np.zeros((R, n_envs), dtype=np.float32)
            for t in range(R):
                ob_s[t] = np.stack(states)
                x = torch.from_numpy(np.stack(states)).float().to(DEVICE) / 255.0
                with torch.inference_mode():
                    logits, v = net(x)
                    dist = Categorical(logits=logits)
                    a = dist.sample()
                    lp = dist.log_prob(a)
                acts[t] = a.cpu().numpy()
                logps[t] = lp.cpu().numpy()
                vals[t] = v.cpu().numpy()
                for i, env in enumerate(envs):
                    o, reward, done, score, discount, adv_frames = env.step(int(acts[t, i]))
                    rews[t, i] = reward
                    discs[t, i] = discount
                    dones[t, i] = float(done)
                    total_frames += adv_frames
                    stacks[i].append(o)
                    states[i] = stack_frames(stacks[i])
                    if done:
                        episode_counter += 1
                        if score > best_score:
                            best_score = score
                            torch.save({'model': net.state_dict(), 'score': score,
                                        'episode': episode_counter, 'algo': 'ppo',
                                        'frame_stack': 4, 'obs_res': 80},
                                       out / 'best.pt')
                        o = env.reset(SEED + episode_counter)
                        stacks[i] = deque([o.copy() for _ in range(4)], maxlen=4)
                        states[i] = stack_frames(stacks[i])
            x = torch.from_numpy(np.stack(states)).float().to(DEVICE) / 255.0
            with torch.inference_mode():
                _, v_last = net(x)
            last_values = v_last.cpu().numpy().astype(np.float32) * (1.0 - dones[-1])
            adv, ret = compute_gae(rews, vals, discs, dones, last_values, gae_lambda)
            adv_t = torch.from_numpy(adv.reshape(-1)).to(DEVICE)
            ret_t = torch.from_numpy(ret.reshape(-1)).to(DEVICE)
            ob_t = torch.from_numpy(ob_s.reshape(-1, 4, 80, 80)).float().to(DEVICE) / 255.0
            act_t = torch.from_numpy(acts.reshape(-1)).to(DEVICE)
            logp_t = torch.from_numpy(logps.reshape(-1)).to(DEVICE)
            adv_t = (adv_t - adv_t.mean()) / (adv_t.std() + 1e-8)
            n_samples = ob_t.shape[0]
            idx = np.arange(n_samples)
            for _ in range(epochs):
                np.random.shuffle(idx)
                for s in range(0, n_samples, minibatch):
                    mb = idx[s:s + minibatch]
                    logits, v = net(ob_t[mb])
                    dist = Categorical(logits=logits)
                    new_logp = dist.log_prob(act_t[mb])
                    ratio = (new_logp - logp_t[mb]).exp()
                    surr1 = ratio * adv_t[mb]
                    surr2 = torch.clamp(ratio, 1 - clip, 1 + clip) * adv_t[mb]
                    policy_loss = -torch.min(surr1, surr2).mean()
                    value_loss = ((v - ret_t[mb]) ** 2).mean()
                    entropy = dist.entropy().mean()
                    loss = policy_loss + value_coef * value_loss - entropy_coef * entropy
                    optimizer.zero_grad(set_to_none=True)
                    loss.backward()
                    torch.nn.utils.clip_grad_norm_(net.parameters(), max_grad_norm)
                    optimizer.step()
            updates += 1
            if updates % 10 == 0:
                print(f'update={updates} frames={total_frames} best={best_score:.0f} '
                      f'ep={episode_counter} ({time.monotonic()-started:.0f}s)')
        torch.save({'model': net.state_dict(), 'score': best_score, 'updates': updates,
                    'algo': 'ppo', 'frame_stack': 4, 'obs_res': 80}, out / 'final.pt')
        print(f'PPO training done. best={best_score:.0f} episodes={episode_counter}')
    finally:
        for e in envs:
            e.close()


if RUN_PPO_TRAINING:
    train_ppo()
else:
    # Load pretrained PPO checkpoint into the matching PPOVisionNet (not GitHubDinoNet)
    PPO_CHECKPOINT = Path('results/exp/ppo_vision/best.pt')
    if PPO_CHECKPOINT.is_file():
        saved = torch.load(PPO_CHECKPOINT, map_location=DEVICE, weights_only=False)
        ppo_net = PPOVisionNet(in_channels=4).to(DEVICE)
        ppo_net.load_state_dict(saved['model'] if isinstance(saved, dict) and 'model' in saved else saved)
        ppo_net.eval().requires_grad_(False)
        print(f'Loaded pretrained PPO checkpoint: {PPO_CHECKPOINT.name}, score={saved.get("score", "?")}')
    else:
        print(f'PPO checkpoint not found: {PPO_CHECKPOINT}')

### GameScaling Training

GameScaling uses a different training paradigm: collect expert demonstrations from the trained DQN, then fine-tune the VLM with LoRA (behavior cloning).

**Step 1: Collect expert demonstrations**

In [ ]:
# Expert demonstration collection (real code, copied from gamescaling/collect_expert.py)
# The DQN checkpoint plays the game; (frame, 200ms key-chunk) pairs are recorded
# aligned to GameScaling's official action format at 12-frame (200ms) boundaries.
RUN_EXPERT_COLLECTION = False

def game_capture_jpeg(game, quality=90):
    """Capture current canvas as JPEG bytes (PIL Image, like GameEnv.capture())."""
    encoded = game.driver.execute_script(
        "return document.querySelector('canvas.runner-canvas').toDataURL('image/png').split(',')[1]")
    img = PILImage.open(io.BytesIO(base64.b64decode(encoded))).convert('RGB')
    buf = io.BytesIO()
    img.save(buf, format='JPEG', quality=quality)
    return buf


def collect_expert(episodes=3, seed_offset=30000, max_frames=4000,
                   out_path='results/expert_data.jsonl', checkpoint='dino_model.pt'):
    net = GitHubDinoNet()
    saved = torch.load(io.BytesIO(Path(checkpoint).read_bytes()), map_location='cpu', weights_only=True)
    net.load_state_dict(saved['model'])
    net.eval().requires_grad_(False)
    print('expert checkpoint loaded:', saved.get('episodes', '?'), 'episodes', flush=True)
    out = Path(out_path)
    out.parent.mkdir(parents=True, exist_ok=True)
    total = 0
    t0 = time.monotonic()
    with out.open('w') as f:
        for ep in range(episodes):
            seed = SEED + seed_offset + ep
            game = LocalGitHubGame(SEED, action_repeat=4)
            observation = game.reset(seed)
            frames = [observation.copy() for _ in range(4)]
            done = False
            game_frames = 0
            pending = None
            n_ep = 0
            while not done and game_frames < max_frames:
                boundary = game_frames % 12 == 0
                if boundary:
                    if pending is not None:
                        pending['groups'] = ';'.join(pending['groups'])
                        f.write(json.dumps(pending) + '\n')
                        n_ep += 1
                        total += 1
                    buf = game_capture_jpeg(game)
                    pending = {'seed': seed,
                               'img': base64.b64encode(buf.getvalue()).decode(),
                               'groups': ['', '', '', '', '', '']}
                stack = stack_frames(deque(frames, maxlen=4))
                with torch.inference_mode():
                    a = int(net(torch.from_numpy(stack).float().unsqueeze(0)).argmax(1).item())
                obs, _, done, score, _, adv = game.step(a)
                frames.append(obs)
                game_frames += adv
                if pending is not None:
                    g = game_frames // 2
                    for gi in [g - 2, g - 1]:
                        if 0 <= gi < 6:
                            pending['groups'][gi] = 'Up' if a == 1 else ''
            game.close()
            print(f'episode {ep} seed={seed}: frames={game_frames} score={score:.0f} '
                  f'samples={n_ep} done={done}', flush=True)
    print(f'total samples: {total} -> {out} ({time.monotonic()-t0:.0f}s)', flush=True)


if RUN_EXPERT_COLLECTION:
    collect_expert()
else:
    print('Expert demonstrations: set RUN_EXPERT_COLLECTION=True to collect')

**Step 2: LoRA SFT (behavior cloning)**

In [ ]:
# Step 2: LoRA SFT (behavior cloning) — real code, copied from gamescaling/sft_lora.py
# Frozen backbone + LoRA adapter keeps memory small; loss on action tokens only.
RUN_LORA_SFT = False

def make_lora(model):
    from peft import LoraConfig, get_peft_model
    names = {n.split('.')[-1] for n, _ in model.named_modules()}
    for cand in [['q_proj', 'k_proj', 'v_proj', 'o_proj'],
                 ['q_proj', 'v_proj'],
                 ['query_key_value']]:
        if all(c in names for c in cand):
            targets = cand
            break
    else:
        raise SystemExit(f'unknown target modules; sample names={sorted(list(names))[:40]}')
    lcfg = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05, bias='none',
                      task_type='CAUSAL_LM', target_modules=targets)
    peft_model = get_peft_model(model, lcfg)
    peft_model.print_trainable_parameters()
    return peft_model


def sft_lora(data_path='results/expert_data.jsonl', model_dir='gamescaling/model',
             epochs=1.0, lr=2e-4, per_seed=250, max_len=2400, accum=4,
             seed=7, limit=0, out_dir='results/sft_lora'):
    import math, random
    from transformers import AutoModelForMultimodalLM, AutoProcessor
    sys.path.insert(0, str(Path('gamescaling').resolve()))
    from gamescaling.action import ACTION_END, ACTION_START, THINK_OPEN
    from gamescaling.policy import NO_THINK_PREFILL, assistant_turn, user_turn
    from gamescaling.gamescaling_dino import build_dino_prompt

    def action_text_from_groups(groups):
        body = '0 0 0 ; ' + ' ; '.join(groups)
        return f'{ACTION_START}{body}{ACTION_END}'

    random.seed(seed)
    torch.manual_seed(seed)
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    dtype = torch.bfloat16 if device.type == 'cuda' else torch.float32

    print('loading processor / model ...', flush=True)
    processor = AutoProcessor.from_pretrained(model_dir)
    model = AutoModelForMultimodalLM.from_pretrained(model_dir, dtype=dtype).to(device)
    model.gradient_checkpointing_enable()
    model.config.use_cache = False
    model.enable_input_require_grads()
    model.eval()
    lora = make_lora(model)
    lora.to(device)

    rows = []
    with open(data_path) as f:
        for line in f:
            r = json.loads(line)
            r['groups_list'] = r['groups'].split(';')
            r['action_text'] = action_text_from_groups(r['groups_list'])
            r['has_jump'] = 'Up' in r['groups']
            rows.append(r)
    by_seed = {}
    for r in rows:
        by_seed.setdefault(r['seed'], []).append(r)
    n_rows = sum(len(v) for v in by_seed.values())
    n_pos = sum(r['has_jump'] for v in by_seed.values() for r in v)
    print(f'data: {n_rows} rows across {len(by_seed)} seeds, {n_pos} jump rows', flush=True)

    rng = random.Random(seed)
    items = []
    w_probs = {1: 0.4, 2: 0.25, 3: 0.15, 4: 0.1, 6: 0.1}
    for ep_seed, rows_ in by_seed.items():
        pool = []
        i = 0
        while i < len(rows_):
            w = rng.choices(list(w_probs), list(w_probs.values()))[0]
            w = min(w, len(rows_) - i)
            pool.append(rows_[i:i + w])
            i += w
        rng.shuffle(pool)
        for window in pool[:per_seed]:
            items.append(window)
            if window[-1]['has_jump']:
                items.extend([window] * 3)
    rng.shuffle(items)
    print(f'windows: {len(items)}', flush=True)

    def render_sample(window):
        history = window[:-1]
        target = window[-1]
        messages = [{'role': 'system', 'content': build_dino_prompt()}]
        url_prefix = 'data:image/jpeg;base64,'
        for row in history:
            messages.append(user_turn(url_prefix + row['img']))
            messages.append(assistant_turn(row['action_text']))
        messages.append(user_turn(url_prefix + target['img']))
        messages.append({'role': 'assistant',
                         'content': NO_THINK_PREFILL + target['action_text']})
        inputs = processor.apply_chat_template(messages, tokenize=True,
                                               return_dict=True, return_tensors='pt')
        ids = inputs['input_ids'][0]
        tokz = processor.tokenizer
        tid = tokz.convert_tokens_to_ids(THINK_OPEN)
        seq_ids = [tid] if (tid is not None and isinstance(tid, int) and tid >= 0) \
            else tokz(THINK_OPEN, add_special_tokens=False).input_ids
        last_think = None
        for i in range(len(ids) - len(seq_ids), -1, -1):
            if ids[i:i + len(seq_ids)] == torch.tensor(seq_ids):
                last_think = i + len(seq_ids) - 1
                break
        return inputs, last_think

    opt = torch.optim.AdamW([p for p in lora.parameters() if p.requires_grad],
                            lr=lr, weight_decay=0.0)
    steps_per_epoch = math.ceil(len(items) / accum)
    total_steps = int(steps_per_epoch * epochs)
    warmup = 20
    sched = torch.optim.lr_scheduler.LambdaLR(
        opt, lambda s: min((s + 1) / warmup, 1.0) * 0.5 * (1 + math.cos(
            math.pi * min(max(s - 0, 0), total_steps) / max(total_steps - 1, 1))))
    print(f'steps/epoch={steps_per_epoch} total_steps={total_steps}', flush=True)

    losses, step, micro, done, epoch = [], 0, 0, False, 0
    t_start = time.monotonic()
    while not done:
        epoch += 1
        for w in items:
            inputs, last_think = render_sample(w)
            if last_think is None:
                continue
            L = inputs['input_ids'].shape[-1]
            if L > max_len:
                continue
            inputs = inputs.to(device)
            take = L - (last_think + 1)
            if take <= 0:
                continue
            out = lora(**inputs, logits_to_keep=take + 1)
            logits = out.logits[:, :-1, :].float()
            targets = inputs['input_ids'][:, -take:]
            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]),
                                   targets.reshape(-1))
            (loss / accum).backward()
            torch.nn.utils.clip_grad_norm_(
                [p for p in lora.parameters() if p.requires_grad], 1.0)
            losses.append(float(loss.detach()) * accum)
            micro += 1
            if limit:
                if micro >= limit:
                    done = True
                    break
            if micro % accum == 0:
                opt.step(); opt.zero_grad(set_to_none=True); sched.step()
                step += 1
                if step % 20 == 0:
                    rate = micro / (time.monotonic() - t_start + 1e-9)
                    eta_min = (len(items) * epochs - micro) / max(rate, 1e-9) / 60
                    print(f'step {step}/{total_steps} epoch {epoch} '
                          f'loss={sum(losses[-80:])/min(80,len(losses)):.4f} '
                          f'window/s={rate:.2f} eta={eta_min:.0f}min', flush=True)
                if step >= total_steps:
                    done = True
                    break
        if done:
            break
    loss_final = sum(losses) / max(1, len(losses))
    print(f'mean micro loss over run: {loss_final:.4f}', flush=True)
    if not limit:
        outdir = Path(out_dir)
        outdir.mkdir(parents=True, exist_ok=True)
        lora.save_pretrained(outdir)
        print(f'adapter saved -> {outdir}', flush=True)


if RUN_LORA_SFT:
    sft_lora()
else:
    print('LoRA SFT: set RUN_LORA_SFT=True to train (needs gamescaling/model + peft + transformers)')

### Training Results

The training curve shows scores rising from 50 to the perfect 2995:

In [ ]:
# Training log: display key metrics from training CSV files
import csv
from pathlib import Path

def show_training_log(path, name):
    p = Path(path)
    if not p.exists():
        print(f'  {name}: no log found at {p}')
        return
    rows = list(csv.DictReader(open(p)))
    if not rows:
        print(f'  {name}: empty log')
        return
    scores = [float(r['score']) for r in rows if 'score' in r]
    best = max(scores) if scores else 0
    final_ma = sum(scores[-50:]) / min(50, len(scores)) if scores else 0
    total_frames = rows[-1].get('total_frames', rows[-1].get('frames', '?'))
    print(f'  {name:20s}  episodes={len(rows):5d}  best={best:7.1f}  '
          f'final_ma(50)={final_ma:7.1f}  total_frames={total_frames}')

print('=== Training Logs ===')
show_training_log('results/training_log.csv', 'DDQN (train_fast)')
show_training_log('results/exp/dqn_vision/training_log.csv', 'DQN (vision)')
show_training_log('results/exp/c51_vision/training_log.csv', 'C51 (vision)')
show_training_log('results/exp/ppo_vision/training_log.csv', 'PPO (vision)')
show_training_log('results/exp/state_dqn/training_log.csv', 'State DQN')

print()
print('=== Ablation Training Logs ===')
for name in ['abl_ppo_noent', 'abl_ppo_nogae', 'abl_ppo_nonorm', 'abl_ppo_clip01',
             'abl_obs_stack1', 'abl_obs_stack2', 'abl_obs_res64',
             'abl_rew_nostep', 'abl_rew_nopenalty', 'abl_rew_sparse']:
    show_training_log(f'results/exp/{name}/training_log.csv', name)

**Training process diagrams:**

In [ ]:
from IPython.display import Image, display
display(Image('results/pptx_images/slide11_3.png'))

In [ ]:
from IPython.display import Image, display
display(Image('results/pptx_images/slide12_4.png'))

In [ ]:
from IPython.display import Image, display
display(Image('results/training_curve.png'))

Trained agent gameplay demo:

In [ ]:
from IPython.display import Image, display
display(Image('results/5716.gif'))

## 4. Experiment Analysis

### Comparison Design

7 methods compared under the same protocol:

| Method | Observation | Type |
|--------|------------|------|
| random | state | no learning |
| rule | state | heuristic |
| DQN | vision | value-based |
| DDQN | vision | value-based |
| C51 | vision | value-based |
| PPO | vision | policy gradient |
| state_dqn | state | value-based (privileged) |

Unified protocol: same action space, reward, 2.23M frame budget, 50 fixed eval seeds under greedy policy.

### Comparison Results

| Method | Greedy Mean | Median | Note |
|--------|-----------|--------|------|
| PPO | **2995** | 2995 | only perfect learning method |
| rule | 2995 | 2995 | rule baseline |
| state_dqn | 1905 | 1905 | privileged state |
| DQN | 66 | 61 | vision off-policy |
| DDQN | 56 | 55 | vision off-policy |
| random | 55 | 52 | random baseline |
| C51 | 46 | 45 | vision off-policy |

PPO is the only successful vision RL method. Vision off-policy methods all fail (worse than random).

### GameScaling VLM Comparison

| Policy | Mean Score | Mean Act Latency | Note |
|--------|-----------|-----------------|------|
| DQN (specialised RL) | **2995** | <1 ms | ms-level decisions |
| GameScaling zero-shot | 42.5 | ~6.4 s | crashes at first obstacle, never jumps |
| GameScaling + LoRA SFT (real-time) | 52.6 | ~1.0 s | learns to jump via behavior cloning |
| GameScaling + LoRA SFT (slowdown=0.15) | **130.3** | ~1.0 s | best mean after decision-interval tuning |
| GameScaling + LoRA SFT (best single episode) | **206** | ~1.0 s | one run under slowdown=0.50 |

**What the completed runs show.**

Behavior cloning from expert demonstrations clearly changes the policy.
Zero-shot GameScaling almost never jumps and dies at the first obstacle (mean 42.5).
After LoRA SFT on expert data that contains jump labels, the model does jump:
the real-time mean rises to 52.6 (scores 61 / 42 / 59 / 50 / 51).

The larger change comes from matching the decision interval to the VLM.
Each action covers 12 game frames (200 ms of game time), while a forward pass
takes about one second on GPU. Adding a fixed slowdown after every step lets
the model finish thinking before the obstacle arrives. Across the completed
slowdown sweeps, the mean rises from 52.6 in real time to about 130
(e.g. slowdown=0.15 → mean 130.3 from scores 173 / 110 / 108;
slowdown=0.70 → mean 131.25; best single episode 206).
Larger slowdowns do not keep improving the mean — the curve flattens and
becomes noisy rather than climbing without bound.

**Conclusion.** GameScaling + LoRA learns *when* to jump, and pacing the
environment recovers a large part of the latency penalty (42.5 → ~130).
The remaining gap to DQN (2995) is still dominated by decision rate:
a 0.8B VLM needs ~1000 ms per decision, while the game advances 12 frames
in that window and a jump arc lasts only ~34 frames — only a few decisions
per jump. Fine-tuning improves the policy; it does not turn the VLM into
a millisecond controller.

> **Note on the bug fix:** the initial expert data collection had an index-overflow bug in
> the action-group recorder — only the first 12 frames of each episode recorded correctly,
> producing zero jump labels. After fixing `g = ((game_frames % 12) or 12) // 2` and
> re-collecting, 25.6% of samples carry jump actions. The LoRA SFT results above use the corrected data.


In [ ]:
# GameScaling LoRA training log + completed evaluation summary
print('=== GameScaling LoRA SFT Training Log ===')
print()
print('Expert data collection (collect_expert.py, fixed action recorder):')
print('  episodes=5, seed_offset=30000, max_frames=20000')
print('  total samples: 5,628 rows across 5 seeds')
print('  jump rows: 1,439 (25.6%)  <- after fixing group index overflow bug')
print()
print('LoRA SFT (sft_lora.py):')
print('  model: GameScaling-0.8B-S3 (854M params, frozen backbone)')
print('  LoRA: r=16, alpha=32, dropout=0.05, target=q/k/v/o_proj (1.08M trainable, 0.13%)')
print('  data: 2,243 windows from 5,628 demonstrations')
print('  optimizer: AdamW lr=2e-4, warmup=20 steps, cosine decay')
print('  steps: 561 total (1 epoch), batch accumulation=4')
print('  training: CUDA (RTX 4090 D), bf16, gradient checkpointing')
print('  loss: 3.28 -> 0.09 (mean 0.28), ~2.3 windows/s')
print('  adapter saved -> results/sft_lora/ (4.3MB)')
print()
print('=== Evaluation: real-time vs slowdown (completed runs) ===')
print()
print('Bottleneck: VLM inference ~1s vs 200ms decision window (12 game frames).')
print('Slowdown sleeps after each step so the model can finish before the obstacle arrives.')
print()
print(f'{"Config":<36} {"Scores":<24} {"Mean":>8}')
print('-' * 72)
print(f'{"Zero-shot (no LoRA)":<36} {"42.5":<24} {"42.5":>8}')
print(f'{"LoRA, real-time":<36} {"61/42/59/50/51":<24} {"52.6":>8}')
print(f'{"LoRA, slowdown=0.15":<36} {"173/110/108":<24} {"130.3":>8}')
print(f'{"LoRA, slowdown=0.50":<36} {"142/59/79/206":<24} {"121.5":>8}')
print(f'{"LoRA, slowdown=0.70":<36} {"111/179/93/142":<24} {"131.25":>8}')
print(f'{"DQN expert reference":<36} {"2995 (cap)":<24} {"2995":>8}')
print()
print('Completed-sweep means (same LoRA adapter, temperature 0):')
print('  slowdown 0.15/0.25/0.35/0.50/0.70 -> 106.2 / 93.4 / 83.6 / 121.5 / 131.25')
print('  slowdown 0.80 / 1.20 / 2.00      -> 130.0 / 89.8 / 109.7')
print('  best mean ~131, best single episode 206')
print()
print('Results:')
print('  - Zero-shot -> real-time LoRA: 42.5 -> 52.6  — BC teaches jumping')
print('  - Real-time -> slowdown LoRA: 52.6 -> ~130   — decision pacing recovers latency')
print('  - Slowdown curve flattens around 130; higher delay does not keep rising')
print('  - DQN stays at 2995 under the same game cap')
print('  - Conclusion: behavior cloning works; inference speed / decision rate')
print('    is the limiting factor for the VLM policy')


**GameScaling evaluation code:**

In [ ]:
# GameScaling zero-shot evaluation (real code, copied from gamescaling/gamescaling_dino.py)
# Renders canvas, sends prompt + history, executes each 200ms action chunk as 12 frames.
RUN_GAMESCALING_EVAL = False

def build_dino_prompt():
    rules = (
        "You are playing the Chrome T-Rex Runner, the offline dinosaur game. The dinosaur "
        "runs automatically from left to right at a constant speed; it cannot stop or "
        "turn.\n"
        "## What you see on screen\n"
        "- The dinosaur stands on the left of the screen and is on the ground unless it "
        "is jumping.\n"
        "- Obstacles enter from the right: single or clustered cacti of different heights "
        "on the ground, and pterodactyls flying at low or high altitude.\n"
        "- The distance score is shown in the top-right corner.\n"
        "## Rules\n"
        "- The run ends the moment the dinosaur touches an obstacle; otherwise the score "
        "grows with distance.\n"
        "- Jump over ground cacti: press the jump key shortly before the dinosaur reaches "
        "the obstacle, and land before the next obstacle of a cluster.\n"
        "- Pterodactyls fly at two heights: jump over the low ones, and keep running "
        "under the high ones without jumping.\n"
        "- Jumping too early makes you land on the obstacle; jump only when it is close.\n"
        "- The speed is constant; ignore the score text and react only to the obstacles."
    )
    sys.path.insert(0, str(Path('gamescaling').resolve()))
    from gamescaling.policy import build_system_prompt
    return build_system_prompt(
        "Chrome T-Rex Runner",
        rules=rules,
        key_bindings=[("Up", "Jump"), ("Space", "Jump (keyboard alternative)")],
        mouse_aim=False,
    )


def run_gamescaling_episode(policy, game, seed, max_decisions=200, max_seconds=300):
    from gamescaling.action import parse_action
    JUMP_KEYS = {"Up", "Space"}
    policy.reset()
    game.reset(seed)
    started = time.monotonic()
    decisions, game_frames, total_reward, score, done = 0, 0, 0.0, 0.0, False
    latencies, unparsed = [], 0
    while (not done and decisions < max_decisions
           and time.monotonic() - started < max_seconds):
        frame = game.capture()
        text, action, ms = policy.act(frame)
        latencies.append(ms)
        if not action.parsed:
            unparsed += 1
        for group in action.steps:
            jump = 1 if JUMP_KEYS.intersection(group) else 0
            _, reward, d, score, _, adv = game.step(jump)
            total_reward += reward
            game_frames += adv
            done = d
            if done:
                break
        decisions += 1
        if decisions % 50 == 0:
            print(f"    decision {decisions}: score={score:.0f} frames={game_frames} "
                  f"last_act={ms:.0f}ms", flush=True)
    terminal = ("collision" if done else "decision_limit")
    result = dict(seed=seed, score=float(score), game_frames=game_frames,
                  decisions=decisions, total_reward=round(total_reward, 2),
                  terminal=terminal, wall_s=round(time.monotonic() - started, 1),
                  mean_act_ms=round(float(np.mean(latencies)), 1) if latencies else None,
                  unparsed=unparsed)
    print('episode result:', result, flush=True)
    return result


def gamescaling_eval(model_dir='gamescaling/model', episodes=5, seed_offset=90000):
    from transformers import AutoModelForMultimodalLM, AutoProcessor
    from collections import deque
    sys.path.insert(0, str(Path('gamescaling').resolve()))
    from gamescaling.action import ACTION_END
    from gamescaling.policy import (NO_THINK_PREFILL, PolicyConfig, assistant_turn,
                                    finalize, parse_action, split_reasoning, user_turn)
    from gamescaling.gamescaling_dino import encode_image
    from torch.nn.utils.rnn import pad_sequence

    class StopOnToken:
        def __init__(self, token_id):
            self.token_id = token_id
        def __call__(self, input_ids, scores, **kwargs):
            return bool(input_ids[0][-1] == self.token_id)

    class LocalGameScalingPolicy:
        def __init__(self, system_prompt, cfg, device='cpu'):
            self.system_prompt = system_prompt
            self.cfg = cfg
            self.device = torch.device(device)
            self.processor = AutoProcessor.from_pretrained(model_dir)
            self.model = AutoModelForMultimodalLM.from_pretrained(model_dir, dtype='auto')
            if device != 'cpu':
                self.model = self.model.to(device)
            self.model.eval()
            self._history = deque()
            end_id = self.processor.tokenizer.convert_tokens_to_ids(ACTION_END)
            self._stop_id = end_id

        def reset(self):
            self._history.clear()

        def messages(self, image_url):
            out = [{'role': 'system', 'content': self.system_prompt}]
            n = len(self._history)
            for i, (url, reasoning, action_text) in enumerate(self._history):
                out.append(user_turn(url))
                age = n - i
                keep = self.cfg.history_think < 0 or age <= self.cfg.history_think
                out.append(assistant_turn(action_text, reasoning if keep else ''))
            out.append(user_turn(image_url))
            out.append({'role': 'assistant', 'content': self.cfg.prefill})
            return out

        @torch.inference_mode()
        def act(self, image):
            url = encode_image(image, self.cfg)
            inputs = self.processor.apply_chat_template(
                self.messages(url), tokenize=True, return_dict=True, return_tensors='pt')
            inputs = {k: v.to(self.device) for k, v in inputs.items()}
            prompt_len = inputs['input_ids'].shape[-1]
            t0 = time.perf_counter()
            out = self.model.generate(
                **inputs, max_new_tokens=self.cfg.max_tokens,
                do_sample=self.cfg.temperature > 0,
                temperature=max(self.cfg.temperature, 1e-4),
                stopping_criteria=None,
                eos_token_id=self._stop_id,
                pad_token_id=self.processor.tokenizer.eos_token_id)
            latency_ms = (time.perf_counter() - t0) * 1000
            new_ids = out[0][prompt_len:]
            finished = bool(new_ids.shape[0]) and bool(new_ids[-1] == self._stop_id)
            text = self.processor.tokenizer.decode(new_ids, skip_special_tokens=False)
            text = finalize(text, 'stop' if finished else 'length', self.cfg)
            action = parse_action(text, lenient=self.cfg.lenient_parse)
            reasoning, _, closed = split_reasoning(text)
            self._history.append((url, reasoning if closed else '', action.history_text))
            while len(self._history) > self.cfg.history_len:
                self._history.popleft()
            return text, action, latency_ms

    cfg = PolicyConfig()
    policy = LocalGameScalingPolicy(build_dino_prompt(), cfg, device='cpu')
    game = LocalGitHubGame(SEED)
    results = []
    for ep in range(episodes):
        print(f'=== episode {ep} ===', flush=True)
        results.append(run_gamescaling_episode(policy, game, SEED + seed_offset + ep))
    game.close()
    scores = [r['score'] for r in results]
    print(f'GameScaling eval done: mean={np.mean(scores):.0f} '
          f'scores={[int(s) for s in scores]}')
    return results


if RUN_GAMESCALING_EVAL:
    gamescaling_eval()
else:
    print('GameScaling zero-shot eval: set RUN_GAMESCALING_EVAL=True (needs gamescaling/model)')

**Expert demonstration collection code:**

In [ ]:
# collect_expert.py: DQN plays and records demonstrations (real runner)
# Output: results/expert_data.jsonl with (frame, action) pairs
# Usage equivalent: python gamescaling/collect_expert.py --episodes 8 --seed-offset 40000 --max-frames 20000
RUN_COLLECT_EXPERT = False

if RUN_COLLECT_EXPERT:
    collect_expert(episodes=8, seed_offset=40000, max_frames=20000,
                   out_path='results/expert_data.jsonl')
else:
    print('Expert collection: set RUN_COLLECT_EXPERT=True '
          '(uses collect_expert() defined in the previous cell)')

**7 methods comparison bar chart:**

In [ ]:
# 7-method comparison bar chart (generated from eval_summary.csv)
import csv
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

EXP = Path('results/exp')
METHODS = ['random', 'rule', 'ddqn', 'dqn', 'c51', 'ppo_vision', 'state_dqn']
LABELS = ['Random', 'Rule', 'DDQN', 'DQN', 'C51', 'PPO (vision)', 'State DQN']

def read_summary(name):
    p = EXP / name / 'eval_summary.csv'
    if not p.exists(): return None
    with open(p) as f: rows = list(csv.DictReader(f))
    return rows[-1] if rows else None

means, stds, maxs = [], [], []
for name in METHODS:
    r = read_summary(name)
    means.append(float(r['mean']) if r else 0)
    stds.append(float(r['std']) if r else 0)
    maxs.append(float(r['max']) if r else 0)

x = np.arange(len(LABELS))
fig, ax = plt.subplots(figsize=(10, 6))
ax.bar(x, means, yerr=stds, capsize=5, color='steelblue', alpha=0.85, label='Greedy eval mean ± std (50 seeds)')
ax.scatter(x, maxs, color='crimson', zorder=3, label='Max score')
ax.axhline(2000, color='green', ls=':', lw=1.2, label='Target 2000')
ax.set_xticks(x); ax.set_xticklabels(LABELS, rotation=25, ha='right')
ax.set_ylabel('Score'); ax.set_title('7-Method Comparison — Greedy Eval on 50 Fixed Seeds')
ax.grid(alpha=0.3, axis='y'); ax.legend(fontsize=9)
for i2, (m, s) in enumerate(zip(means, stds)):
    ax.text(i2, m + s + 60, f'{m:.0f}', ha='center', fontsize=9, fontweight='bold')
plt.tight_layout(); plt.savefig(EXP / 'compare_bar_chart.png', dpi=130); plt.show()
print('Chart saved to results/exp/compare_bar_chart.png')

**Training curves comparison:**

In [ ]:
# Training curves + eval bars (generated from training_log.csv + eval_summary.csv)
import csv
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

EXP = Path('results/exp')
TRAIN_LOGS = {
    'DDQN (train_fast)': Path('results/training_log.csv'),
    'DQN (vision)': EXP / 'dqn_vision' / 'training_log.csv',
    'C51 (vision)': EXP / 'c51_vision' / 'training_log.csv',
    'PPO (vision)': EXP / 'ppo_vision' / 'training_log.csv',
    'State DQN': EXP / 'state_dqn' / 'training_log.csv',
}
EVAL_NAMES = ['random', 'rule', 'ddqn', 'dqn', 'c51', 'ppo_vision', 'state_dqn']
EVAL_LABELS = ['Random', 'Rule', 'DDQN', 'DQN', 'C51', 'PPO', 'State DQN']

def read_csv(p):
    if not p.exists(): return []
    with open(p) as f: return list(csv.DictReader(f))

def moving_avg(x, w=50):
    if len(x) < w: return np.asarray(x, dtype=float)
    return np.convolve(np.asarray(x, dtype=float), np.ones(w)/w, mode='valid')

fig, (ax, ax2) = plt.subplots(1, 2, figsize=(15, 6), gridspec_kw={'width_ratios': [3, 2]})
colors = plt.cm.tab10.colors
for ci, (label, p) in enumerate(TRAIN_LOGS.items()):
    rows = read_csv(p)
    if not rows: continue
    score = np.array([float(r['score']) for r in rows])
    frames = np.array([int(r.get('total_frames', r.get('frames', 0))) for r in rows]) / 1e6
    ma = moving_avg(score, 50)
    ax.plot(frames[49:], ma, lw=1.8, color=colors[ci%10], label=f'{label} (ma {ma[-1]:.0f})')
    ax.plot(frames, np.maximum.accumulate(score), lw=1, ls='--', color=colors[ci%10], alpha=0.5)
ax.axhline(2000, color='green', ls=':', lw=1.2, label='Target 2000')
ax.set_xlabel('Training frames (M)'); ax.set_ylabel('Episode score (50-ep moving avg)')
ax.set_title('Training Curves'); ax.grid(alpha=0.3); ax.legend(fontsize=9)

means, stds = [], []
for name in EVAL_NAMES:
    rr = read_csv(EXP / name / 'eval_summary.csv')
    if rr: means.append(float(rr[-1]['mean'])); stds.append(float(rr[-1]['std']))
    else: means.append(0); stds.append(0)
x = np.arange(len(EVAL_LABELS))
ax2.bar(x, means, yerr=stds, capsize=4, color='steelblue', alpha=0.8)
ax2.set_xticks(x); ax2.set_xticklabels(EVAL_LABELS, rotation=30, ha='right')
ax2.set_ylabel('Greedy eval score'); ax2.set_title('Eval on 50 Seeds'); ax2.grid(alpha=0.3, axis='y')
plt.tight_layout(); plt.savefig(EXP / 'compare_curves.png', dpi=130); plt.show()
print('Chart saved to results/exp/compare_curves.png')

Comparison experiment training code. Set `RUN_COMPARISON = True` to start training:

In [ ]:
# 7-method comparison training — calls train_offpolicy() / train_ppo() inline.
# Set RUN_COMPARISON = True to train all methods from scratch.
RUN_COMPARISON = False

from pathlib import Path
import csv, torch
DEVICE = torch.device('cpu')

if RUN_COMPARISON:
    # Random and rule baselines are evaluated (not trained).
    print('=== Training DQN (vision) ===')
    train_offpolicy(algo='dqn', obs='vision', name='dqn_vision', frames_budget=2_230_000)

    print('=== Training Double-DQN (vision) ===')
    train_offpolicy(algo='ddqn', obs='vision', name='ddqn', frames_budget=2_230_000)

    print('=== Training C51 (vision) ===')
    train_offpolicy(algo='c51', obs='vision', name='c51_vision', frames_budget=2_230_000)

    print('=== Training PPO (vision) ===')
    # Uses the train_ppo function defined earlier (cell 57)
    train_ppo(total_updates=500, rollout_rounds=64, epochs=4, minibatch=256,
              n_envs=4, name='ppo_vision')

    print('=== Training State DQN (privileged) ===')
    train_offpolicy(algo='ddqn', obs='state', name='state_dqn', frames_budget=2_230_000)
else:
    comparison_checkpoints = {
        'dqn':       Path('results/exp/dqn_vision/best.pt'),
        'ddqn':      Path('results/dino_trained_best.pt'),
        'c51':       Path('results/exp/c51_vision/best.pt'),
        'ppo':       Path('results/exp/ppo_vision/best.pt'),
        'state_dqn': Path('results/exp/state_dqn/best.pt'),
    }
    print('Loading existing model checkpoints:')
    for name, ckpt in comparison_checkpoints.items():
        status = 'OK' if ckpt.is_file() else 'NOT FOUND'
        print(f'  {name}: {status} ({ckpt.name})')
    print()
    table_path = Path('results/exp/compare_table.csv')
    if table_path.exists():
        print('Comparison results:')
        with open(table_path) as f:
            for row in csv.DictReader(f):
                print(f"  {row.get('name',''):12s}  mean={row.get('mean',''):>8s}  "
                      f"median={row.get('median',''):>8s}  std={row.get('std',''):>7s}")

### Rule Baseline

The rule heuristic decides jump timing based on obstacle distance and game speed. For a single cactus: jump when dist < speed*12. For dense clusters: jump when dist < speed*7. Jump takes about 34 frames in the air, covering roughly 200 pixels.

In [ ]:
def rule_policy(state, speed=6):
    pass

print('Rule baseline: jump based on obstacle distance and speed')

### Ablation Study Design

Anchors: PPO at 2995 and state-DQN at 1905. Remove one component at a time and measure the drop.

| Axis | Variant | Description | Anchor |
|------|---------|-------------|--------|
| PPO component | noent | remove entropy bonus | ppo_vision |
| PPO component | nogae | remove GAE | ppo_vision |
| PPO component | nonorm | remove advantage normalization | ppo_vision |
| PPO component | clip01 | shrink clip to 0.1 | ppo_vision |
| Observation | stack1 | frame stack 4->1 | ppo_vision |
| Observation | stack2 | frame stack 4->2 | ppo_vision |
| Observation | res64 | resolution 80->64 | ppo_vision |
| Reward | nostep | remove step reward | ppo_vision |
| Reward | nopenalty | remove early penalty | ppo_vision |
| Reward | sparse | remove both | ppo_vision |
| State feature | notype / notrex / noheight / distonly | remove feature groups | state_dqn |

### PPO Component Ablation Results

In [ ]:
# PPO Component Ablation chart (generated from ablation_report.csv)
import csv
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

EXP = Path('results/exp')
report = list(csv.DictReader(open(EXP / 'ablation_report.csv')))
rows = [r for r in report if r['axis'] == 'ppo_component']
anchor = [r for r in report if r['name'] == 'ppo_vision'][0]
anchor_mean = float(anchor['eval_mean'])

labels = ['Anchor'] + [r['name'].replace('abl_ppo_', '') for r in rows]
values = [anchor_mean] + [float(r['eval_mean']) for r in rows]
deltas = [0] + [float(r['delta_vs_anchor']) for r in rows]
colors = ['green'] + ['crimson' if d < -500 else 'orange' if d < -100 else 'steelblue' for d in deltas[1:]]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(labels))
ax.bar(x, values, color=colors, alpha=0.85, edgecolor='gray')
ax.axhline(anchor_mean, color='green', ls='--', lw=1, alpha=0.5, label=f'Anchor = {anchor_mean:.0f}')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right')
ax.set_ylabel('Greedy eval mean (50 seeds)')
ax.set_title('PPO Component Ablation')
ax.grid(alpha=0.3, axis='y')
for j, (v, d) in enumerate(zip(values, deltas)):
    if d != 0: ax.text(j, v+40, f'{v:.0f}\n({d:+.0f})', ha='center', fontsize=8)
    else: ax.text(j, v+40, f'{v:.0f}', ha='center', fontsize=9, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(EXP / 'ablation_ppo_chart.png', dpi=130); plt.show()
print('Chart saved to results/exp/ablation_ppo_chart.png')

GAE is the most critical component: removing it drops the score from 2995 to 744 (delta -2175). Entropy bonus is second (1633). Clip range third (1716). Advantage normalization matters least (2707).


Training code. Set `RUN_ABLATION_PPO = True` to start:

In [ ]:
# PPO component ablation — each variant changes ONE component from the anchor.
# Set RUN_ABLATION_PPO = True to train all variants.
RUN_ABLATION_PPO = False

from pathlib import Path
import csv, torch
DEVICE = torch.device('cpu')

ablation_ppo_configs = {
    'noent':  dict(entropy_coef=0.0,   desc='remove entropy bonus'),
    'nogae':  dict(gae_lambda=1.0,     desc='GAE lambda=1 (pure MC returns)'),
    'nonorm': dict(no_adv_norm=True,   desc='disable advantage normalization'),
    'clip01': dict(clip=0.1,           desc='narrower PPO clip 0.2 -> 0.1'),
}

if RUN_ABLATION_PPO:
    for name, cfg in ablation_ppo_configs.items():
        desc = cfg.pop('desc', '')
        print(f'=== Training abl_ppo_{name}: {desc} ===')
        train_ppo(total_updates=500, rollout_rounds=64, epochs=4, minibatch=256,
                  n_envs=4, name=f'abl_ppo_{name}', **cfg)
else:
    print('Loading existing PPO ablation results:')
    report = list(csv.DictReader(open('results/exp/ablation_report.csv')))
    for name in ['noent', 'nogae', 'nonorm', 'clip01']:
        full = f'abl_ppo_{name}'
        row = next((r for r in report if r['name'] == full), None)
        if row:
            delta = row.get('delta_vs_anchor', '')
            print(f"  {full:22s}  mean={row['eval_mean']:>8s}  Δ={delta:>8s}  "
                  f"train_best={row.get('train_best','')}")

### Observation Ablation Results

In [ ]:
# Observation Ablation chart (generated from ablation_report.csv)
import csv
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

EXP = Path('results/exp')
report = list(csv.DictReader(open(EXP / 'ablation_report.csv')))
rows = [r for r in report if r['axis'] == 'observation']
anchor = [r for r in report if r['name'] == 'ppo_vision'][0]
anchor_mean = float(anchor['eval_mean'])

labels = ['Anchor'] + [r['name'].replace('abl_observation_', '') for r in rows]
values = [anchor_mean] + [float(r['eval_mean']) for r in rows]
deltas = [0] + [float(r['delta_vs_anchor']) for r in rows]
colors = ['green'] + ['crimson' if d < -500 else 'orange' if d < -100 else 'steelblue' for d in deltas[1:]]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(labels))
ax.bar(x, values, color=colors, alpha=0.85, edgecolor='gray')
ax.axhline(anchor_mean, color='green', ls='--', lw=1, alpha=0.5, label=f'Anchor = {anchor_mean:.0f}')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right')
ax.set_ylabel('Greedy eval mean (50 seeds)')
ax.set_title('Observation Ablation')
ax.grid(alpha=0.3, axis='y')
for j, (v, d) in enumerate(zip(values, deltas)):
    if d != 0: ax.text(j, v+40, f'{v:.0f}\n({d:+.0f})', ha='center', fontsize=8)
    else: ax.text(j, v+40, f'{v:.0f}', ha='center', fontsize=9, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(EXP / 'ablation_obs_chart.png', dpi=130); plt.show()
print('Chart saved to results/exp/ablation_obs_chart.png')

Frame stack 4->1 drops 279 points, 4->2 drops 887. Resolution 80->64 drops 694. Four stacked frames provide important motion information.


Training code:

In [ ]:
# Observation ablation — changes frame stack or resolution.
# Set RUN_ABLATION_OBS = True to train all variants.
RUN_ABLATION_OBS = False

from pathlib import Path
import csv, torch
DEVICE = torch.device('cpu')

ablation_obs_configs = {
    'stack1': dict(frame_stack=1, desc='single frame (no temporal history)'),
    'stack2': dict(frame_stack=2, desc='2-frame stack instead of 4'),
    'res64':  dict(obs_res=64,    desc='64x64 resolution instead of 80x80'),
}

if RUN_ABLATION_OBS:
    for name, cfg in ablation_obs_configs.items():
        desc = cfg.pop('desc', '')
        print(f'=== Training abl_obs_{name}: {desc} ===')
        train_offpolicy(algo='ddqn', obs='vision', name=f'abl_obs_{name}',
                        frames_budget=2_230_000, **cfg)
else:
    print('Loading existing observation ablation results:')
    report = list(csv.DictReader(open('results/exp/ablation_report.csv')))
    for name in ['stack1', 'stack2', 'res64']:
        full = f'abl_obs_{name}'
        row = next((r for r in report if r['name'] == full), None)
        if row:
            delta = row.get('delta_vs_anchor', '')
            print(f"  {full:22s}  mean={row['eval_mean']:>8s}  Δ={delta:>8s}  "
                  f"train_best={row.get('train_best','')}")

### Reward Ablation Results

In [ ]:
# Reward Shaping Ablation chart (generated from ablation_report.csv)
import csv
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

EXP = Path('results/exp')
report = list(csv.DictReader(open(EXP / 'ablation_report.csv')))
rows = [r for r in report if r['axis'] == 'reward']
anchor = [r for r in report if r['name'] == 'ppo_vision'][0]
anchor_mean = float(anchor['eval_mean'])

labels = ['Anchor'] + [r['name'].replace('abl_reward_', '') for r in rows]
values = [anchor_mean] + [float(r['eval_mean']) for r in rows]
deltas = [0] + [float(r['delta_vs_anchor']) for r in rows]
colors = ['green'] + ['crimson' if d < -500 else 'orange' if d < -100 else 'steelblue' for d in deltas[1:]]

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(labels))
ax.bar(x, values, color=colors, alpha=0.85, edgecolor='gray')
ax.axhline(anchor_mean, color='green', ls='--', lw=1, alpha=0.5, label=f'Anchor = {anchor_mean:.0f}')
ax.set_xticks(x); ax.set_xticklabels(labels, rotation=20, ha='right')
ax.set_ylabel('Greedy eval mean (50 seeds)')
ax.set_title('Reward Shaping Ablation')
ax.grid(alpha=0.3, axis='y')
for j, (v, d) in enumerate(zip(values, deltas)):
    if d != 0: ax.text(j, v+40, f'{v:.0f}\n({d:+.0f})', ha='center', fontsize=8)
    else: ax.text(j, v+40, f'{v:.0f}', ha='center', fontsize=9, fontweight='bold')
ax.legend(fontsize=9)
plt.tight_layout(); plt.savefig(EXP / 'ablation_rew_chart.png', dpi=130); plt.show()
print('Chart saved to results/exp/ablation_rew_chart.png')

Removing step reward drops 970 points. Removing early-crash penalty drops 1274. Removing both (sparse) still reaches 2995 -- reward shaping helps but is not strictly necessary.


Training code:

In [ ]:
# Reward ablation — removes one or both reward-shaping terms.
# Set RUN_ABLATION_REW = True to train all variants.
RUN_ABLATION_REW = False

from pathlib import Path
import csv, torch
DEVICE = torch.device('cpu')

ablation_rew_configs = {
    'nostep':    dict(step_reward=0.0,                  desc='no +0.1/step shaping'),
    'nopenalty': dict(early_penalty=0.0,                desc='no -5 early-crash penalty'),
    'sparse':    dict(step_reward=0.0, early_penalty=0.0, desc='pure sparse: only -1 collision'),
}

if RUN_ABLATION_REW:
    for name, cfg in ablation_rew_configs.items():
        desc = cfg.pop('desc', '')
        print(f'=== Training abl_rew_{name}: {desc} ===')
        train_offpolicy(algo='ddqn', obs='vision', name=f'abl_rew_{name}',
                        frames_budget=2_230_000, **cfg)
else:
    print('Loading existing reward ablation results:')
    report = list(csv.DictReader(open('results/exp/ablation_report.csv')))
    for name in ['nostep', 'nopenalty', 'sparse']:
        full = f'abl_rew_{name}'
        row = next((r for r in report if r['name'] == full), None)
        if row:
            delta = row.get('delta_vs_anchor', '')
            print(f"  {full:22s}  mean={row['eval_mean']:>8s}  Δ={delta:>8s}  "
                  f"train_best={row.get('train_best','')}")

### State-Feature Ablation Results

Removing the tRex engine state (y/velocity/jumping) collapses performance to random level (54.9).
Just 4 of 26 dimensions (jumping flag + 3 obstacle distances) actually **outperforms** the full
state vector (2788.9 vs 1879.5) — the extra features are noise for the MLP.
Obstacle-type one-hots carry more useful information than continuous width/height/top.

In [ ]:
# State-feature ablation chart (generated from ablation_report.csv)
import csv
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

EXP = Path('results/exp')
report = list(csv.DictReader(open(EXP / 'ablation_report.csv')))
state_rows = [r for r in report if r['axis'] == 'state_feature']
anchor = [r for r in report if r['name'] == 'state_dqn'][0]
anchor_mean = float(anchor['eval_mean'])

labels = ['Full 26-dim state'] + [r['name'].replace('abl_state_', '') for r in state_rows]
values = [anchor_mean] + [float(r['eval_mean']) for r in state_rows]
deltas = [0] + [float(r['delta_vs_anchor']) for r in state_rows]
colors = ['green'] + ['crimson' if d < -500 else 'orange' if d < -100 else 'steelblue' for d in deltas[1:]]

fig, ax = plt.subplots(figsize=(9, 6))
x = np.arange(len(labels))
ax.bar(x, values, color=colors, alpha=0.85, edgecolor='gray')
ax.axhline(anchor_mean, color='green', ls='--', lw=1, alpha=0.5, label=f'State-DQN anchor = {anchor_mean:.0f}')
ax.set_xticks(x)
ax.set_xticklabels(labels, rotation=15, ha='right')
ax.set_ylabel('Greedy eval mean (50 seeds)')
ax.set_title('State-Feature Ablation — 26-Dim Engine State')
ax.grid(alpha=0.3, axis='y')

for i, (v, d) in enumerate(zip(values, deltas)):
    if d != 0:
        ax.text(i, v + 40, f'{v:.0f}\n({d:+.0f})', ha='center', fontsize=8)
    else:
        ax.text(i, v + 40, f'{v:.0f}', ha='center', fontsize=9, fontweight='bold')

ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(EXP / 'ablation_state_chart.png', dpi=130)
plt.show()
print('Chart saved to results/exp/ablation_state_chart.png')

In [ ]:
# State-feature ablation — masks out feature groups from 26-dim engine state.
# Set RUN_ABLATION_STATE = True to train all variants.
RUN_ABLATION_STATE = False

from pathlib import Path
import csv, torch
DEVICE = torch.device('cpu')

ablation_state_configs = {
    'notype':   dict(feat_mask='notype',   desc='remove obstacle-type one-hots'),
    'notrex':   dict(feat_mask='notrex',   desc='remove tRex y/vy/jumping/ducking/speed'),
    'noheight': dict(feat_mask='noheight', desc='remove obstacle width/height/top'),
    'distonly': dict(feat_mask='distonly', desc='keep only jumping flag + 3 distances'),
}

if RUN_ABLATION_STATE:
    for name, cfg in ablation_state_configs.items():
        desc = cfg.pop('desc', '')
        print(f'=== Training abl_state_{name}: {desc} ===')
        train_offpolicy(algo='ddqn', obs='state', name=f'abl_state_{name}',
                        frames_budget=2_230_000, **cfg)
else:
    print('Loading existing state-feature ablation results:')
    report = list(csv.DictReader(open('results/exp/ablation_report.csv')))
    for name in ['notype', 'notrex', 'noheight', 'distonly']:
        full = f'abl_state_{name}'
        row = next((r for r in report if r['name'] == full), None)
        if row:
            delta = row.get('delta_vs_anchor', '')
            print(f"  {full:22s}  mean={row['eval_mean']:>8s}  Δ={delta:>8s}  "
                  f"train_best={row.get('train_best','')}")

### Conclusions

This project gave us a complete picture of how deep reinforcement learning performs on a real-time visual control task. We built the full pipeline from scratch: a deterministic game environment, a CNN policy network, training loops for multiple algorithms, and a unified evaluation protocol. The main finding is clear — PPO is the only method that reliably solves this task, reaching the perfect score of 2995, while all off-policy value-based methods fail badly under greedy evaluation despite decent training scores.

Our ablation study showed that GAE is the single most important component for PPO. Removing it drops the score from 2995 to 744. Entropy bonus, clip range, and frame stacking all contribute significantly. We also found that reward shaping helps but is not strictly necessary, since pure sparse reward still reaches 2995. Fewer state features can outperform the full feature set.

We also applied a vision-language model through behavior cloning from expert demonstrations. The VLM learned when to jump: the mean rose from 42.5 zero-shot to 52.6 after LoRA SFT, and to about 130 once each decision was given time to finish before the game advanced. That is a large relative gain, and it shows the model did absorb the expert policy. The absolute score still sits far below DQN because each VLM decision takes about a second, while the game moves twelve frames in that interval and a jump only lasts a few decisions. The lesson is that large models can learn a new skill from demonstrations, and that matching the decision clock to the model's speed matters as much as the training loss.

Overall, this project strengthened our understanding of both the capabilities and limitations of current RL methods. We learned that on-policy methods like PPO are more robust for visual control tasks, that ablation studies are essential for understanding which components truly matter, and that rigorous evaluation is the foundation of reliable experimental results. These lessons will guide our future work in reinforcement learning and beyond.


**Final demo:**

In [ ]:
from IPython.display import Image, display
display(Image('results/dino_score_2191.gif'))

### Cleanup

In [ ]:
if 'asset_server' in dir() and asset_server:
    asset_server.shutdown()
    asset_server.server_close()
    print('Server stopped')